# 구글 마운트

In [ ]:
from google.colab import drive
import os

def _drive_ready(mountpoint='/content/drive'):
    """진짜로 Drive 가 붙어 있는가.

    isdir('/content/drive/MyDrive') 만 보면 안 된다 — 마운트가 끊긴 자리에 같은 이름의
    폴더가 남으면 붙지도 않았는데 붙은 것처럼 보이고, 그 뒤 읽기·쓰기가 전부 조용히
    로컬 디스크로 새 나간다(한참 뒤 엉뚱한 데서 FileNotFoundError 로 터진다)."""
    return os.path.ismount(mountpoint) and any(
        os.path.isdir(os.path.join(mountpoint, n)) for n in ('MyDrive', 'My Drive')
    )


def _ensure_drive(mountpoint='/content/drive'):
    """이미 붙어 있으면 넘어가고, 아니면 붙인다. 마운트 자리에 찌꺼기가 남아
    mount 가 거부하는 경우("Mountpoint must not already contain files")까지 복구한다."""
    if _drive_ready(mountpoint):
        return
    try:
        drive.mount(mountpoint)
        return
    except ValueError:
        pass

    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if not os.path.ismount(mountpoint):
        # 마운트가 아니니 여기 있는 건 Drive 원본이 아니라 로컬 찌꺼기다. 지우지 말고
        # 옆으로 치워 둔다 — 이 상태에서 저장된 파일이 있었다면 살릴 수 있게.
        # (rm -rf 로 지우는 방법은 쓰지 않는다. 마운트가 살아 있었다면 원본이 날아간다.)
        stale, n = mountpoint + '_stale', 1
        while os.path.exists(stale):
            n += 1
            stale = f'{mountpoint}_stale{n}'
        os.rename(mountpoint, stale)
        print(f'마운트가 아닌 찌꺼기 폴더를 {stale} 로 치웠습니다.')
    drive.mount(mountpoint, force_remount=True)


_ensure_drive()

# 파일별 통합 파이프라인 (콘솔 로그로 검수)

In [ ]:
import pandas as pd
import os
import re
import unicodedata
import time
from google.colab import drive
# Colab 런타임에 xlsxwriter 가 빠져 있을 때가 있다 — 없을 때만 설치한다.
import importlib.util, subprocess, sys
if importlib.util.find_spec('xlsxwriter') is None:
    print('xlsxwriter 설치 중...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'xlsxwriter'], check=True)
import xlsxwriter

import os

def _drive_ready(mountpoint='/content/drive'):
    """진짜로 Drive 가 붙어 있는가.

    isdir('/content/drive/MyDrive') 만 보면 안 된다 — 마운트가 끊긴 자리에 같은 이름의
    폴더가 남으면 붙지도 않았는데 붙은 것처럼 보이고, 그 뒤 읽기·쓰기가 전부 조용히
    로컬 디스크로 새 나간다(한참 뒤 엉뚱한 데서 FileNotFoundError 로 터진다)."""
    return os.path.ismount(mountpoint) and any(
        os.path.isdir(os.path.join(mountpoint, n)) for n in ('MyDrive', 'My Drive')
    )


def _ensure_drive(mountpoint='/content/drive'):
    """이미 붙어 있으면 넘어가고, 아니면 붙인다. 마운트 자리에 찌꺼기가 남아
    mount 가 거부하는 경우("Mountpoint must not already contain files")까지 복구한다."""
    if _drive_ready(mountpoint):
        return
    try:
        drive.mount(mountpoint)
        return
    except ValueError:
        pass

    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if not os.path.ismount(mountpoint):
        # 마운트가 아니니 여기 있는 건 Drive 원본이 아니라 로컬 찌꺼기다. 지우지 말고
        # 옆으로 치워 둔다 — 이 상태에서 저장된 파일이 있었다면 살릴 수 있게.
        # (rm -rf 로 지우는 방법은 쓰지 않는다. 마운트가 살아 있었다면 원본이 날아간다.)
        stale, n = mountpoint + '_stale', 1
        while os.path.exists(stale):
            n += 1
            stale = f'{mountpoint}_stale{n}'
        os.rename(mountpoint, stale)
        print(f'마운트가 아닌 찌꺼기 폴더를 {stale} 로 치웠습니다.')
    drive.mount(mountpoint, force_remount=True)


_ensure_drive()

# ── 경로 설정
YEAR     = 2022
BASE_DIR = f"/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)"
YEAR_DIR = f"{BASE_DIR}/{YEAR}"
REAL_PATH   = f"{BASE_DIR}/inning_summary/inning_summary_{YEAR}.xlsx"
SUMMARY_OUT = f"{YEAR_DIR}/검수/{YEAR}_enter_inning_summary.xlsx"
DIFF_OUT    = f"{YEAR_DIR}/검수/{YEAR}_diff_inning_summary.xlsx"
ISSUES_OUT  = f"{YEAR_DIR}/검수/{YEAR}_issues.xlsx"  # 오타/이상 항목 목록

os.makedirs(f"{YEAR_DIR}/검수", exist_ok=True)

# ── 공통 스타일 (색은 hex 문자열 — writer 가 xlsxwriter Format 으로 만들어 돌려쓴다)
_C_HEADER = '4472C4'   # 헤더 (파랑)
_C_ISSUES = 'C55A11'   # 헤더 (주황, issues 계열)
_C_EMPH   = 'DCE6F1'   # 강조 열 (file_name/game_id)
_C_GREEN  = 'C6EFCE'   # diff == 0 / 득점
_C_RED    = 'FFD7D7'   # diff != 0
_C_BASE   = 'FFF2CC'   # 주자 있는 루
_C_GRID   = 'BFBFBF'   # 테두리


def _xw_width(w):
    """openpyxl 이 쓰던 열 너비를 그대로 재현한다.

    xlsxwriter 는 엑셀 UI 에 보이는 값으로 받아서 XML 에 여백(5px)을 더해 적고,
    openpyxl 은 받은 값을 XML 에 그대로 적었다. 같은 숫자를 넘기면 새로 만든 파일의 열이
    0.71 씩 넓어져 예전 산출물과 나란히 놓으면 티가 난다 — 여백만큼 빼서 넘긴다."""
    return w - 5 / 7


def write_styled_xlsx(df, path, sheet, headers=None, widths=None, default_width=12,
                      header_color=_C_HEADER, emphasis=(), left=(), zero_green=(),
                      truthy_fill=None, autofilter=False):
    """검수 산출물 엑셀을 한 곳에서 만든다 — 예전 save_*_xlsx 다섯 개의 공통 부분.

    openpyxl 로 셀마다 font/alignment/border 를 대입하면 대입할 때마다 스타일을 새로
    등록해서, 913행짜리 주자상황 파일 하나에 0.7초가 걸렸다(시즌 92파일이면 그 저장에만
    60초가 넘고, box 대조표까지 합치면 100초를 넘긴다). xlsxwriter 는 Format 을 미리
    만들어 돌려쓰므로 같은 모양이 4~5배 빨리 나온다. 산출물의 겉모습(색·테두리·너비·
    틀고정·필터)은 예전 그대로다.

    headers     : 쓸 열과 그 순서. None 이면 df 의 열 전부. df 에 없는 열은 빈칸으로 남긴다.
    widths      : {열: 너비} 또는 headers 순서대로의 리스트. 안 적은 열은 default_width.
    emphasis    : 굵게 + 연파랑으로 강조할 열 (file_name/game_id 처럼 묶어서 보는 열).
    left        : 좌측정렬할 열 (긴 설명문).
    zero_green  : 0 이면 초록, 아니면 빨강으로 칠할 열 (…_diff).
    truthy_fill : {열: 색} — 값이 비어있지 않으면 그 색으로 칠한다.
    autofilter  : 헤더에 필터를 건다.
    """
    cols        = list(headers) if headers is not None else list(df.columns)
    n_row       = len(df)
    truthy_fill = truthy_fill or {}
    if isinstance(widths, (list, tuple)):
        widths = dict(zip(cols, widths))
    widths = widths or {}

    # 열 단위로 미리 파이썬 값으로 뽑아 둔다 — 셀마다 df 에서 꺼내면 그게 또 병목이다.
    data = [df[c].tolist() if c in df.columns else [''] * n_row for c in cols]

    wb = xlsxwriter.Workbook(path, {'constant_memory': True})
    ws = wb.add_worksheet(sheet)
    _base  = {'font_name': 'Arial', 'font_size': 10, 'valign': 'vcenter',
              'border': 1, 'border_color': _C_GRID}
    f_head = wb.add_format(dict(_base, align='center', bold=True,
                                font_color='FFFFFF', bg_color=header_color))

    # 열마다 형식은 많아야 두 가지(평소 / 칠한 것)라 미리 만들어 둔다. 셀마다 만들면
    # xlsxwriter 도 openpyxl 만큼 느려진다 — 빨라지는 이유가 바로 이 재사용이다.
    f_norm, f_alt = [], []
    for c in cols:
        align = 'left' if c in left else 'center'
        if c in emphasis:
            f_norm.append(wb.add_format(dict(_base, align=align, bold=True, bg_color=_C_EMPH)))
            f_alt.append(None)
        elif c in zero_green:          # 평소=초록(0), 대체=빨강(0 아님)
            f_norm.append(wb.add_format(dict(_base, align=align, bg_color=_C_GREEN)))
            f_alt.append(wb.add_format(dict(_base, align=align, bg_color=_C_RED)))
        elif c in truthy_fill:
            f_norm.append(wb.add_format(dict(_base, align=align)))
            f_alt.append(wb.add_format(dict(_base, align=align, bg_color=truthy_fill[c])))
        else:
            f_norm.append(wb.add_format(dict(_base, align=align)))
            f_alt.append(None)

    # 열 너비·틀고정·필터는 데이터보다 먼저 건다 — constant_memory 모드는 이미 흘려보낸
    # 행으로 되돌아가지 못한다.
    for ci, c in enumerate(cols):
        ws.set_column(ci, ci, _xw_width(widths.get(c, default_width)))
    ws.freeze_panes(1, 0)
    if autofilter:
        ws.autofilter(0, 0, n_row, len(cols) - 1)
    ws.write_row(0, 0, cols, f_head)

    for r in range(n_row):
        for ci, c in enumerate(cols):
            v = data[ci][r]
            if hasattr(v, 'item') and not isinstance(v, (str, bytes)):
                v = v.item()           # numpy 스칼라 -> 파이썬 값 (안 그러면 숫자가 문자로 박힌다)
            if v is None or (not isinstance(v, str) and pd.isna(v)):
                v = ''
            fmt = f_norm[ci]
            if c in zero_green:
                fmt = f_norm[ci] if v == 0 else f_alt[ci]
            elif c in truthy_fill and v:
                fmt = f_alt[ci]
            if v == '':
                ws.write_blank(r + 1, ci, None, fmt)
            elif isinstance(v, (bool, int, float, str)):
                ws.write(r + 1, ci, v, fmt)
            else:
                ws.write(r + 1, ci, str(v), fmt)
    wb.close()

# issues DataFrame 컬럼 정의 (모든 step에서 동일하게 사용)
_ISSUE_COLS = ['game_id', '단계', '문제', '상세']

# ════════════════════════════════════════════════════════════════
# Step 1: 쉼표 삭제
#   이슈 반환: 낫아웃인데 쉼표가 남은 행 (game_id, result)
# ════════════════════════════════════════════════════════════════
_remove_always   = ['대수비', '대타']
_prefixes        = ['포구실책', '송구실책', '실책', '포일', '폭투', '보크']
# 국가대표(WBSC) 전용: 실책을 저지른 자리까지 남긴 코드 — '투송실'(투수 송구실책),
# '포포실'(포수 포구실책). KBSA 는 자리 없이 '송구실책'/'포구실책' 으로만 적으므로
# 이 코드가 KBSA 자료에 나오는 일은 없다. 부분일치 검사에 '포실' 만 넣으면 KBSA 의
# '포실'(포수 실책 출루)까지 걸리므로, 반드시 아홉 자리를 모두 적은 집합을 쓴다.
_NT_POS = ('투', '포', '1', '2', '3', '유', '좌', '중', '우')
NT_ERRORS = {f'{_p}{_k}' for _p in _NT_POS for _k in ('송실', '포실')}

_NATOUT_SUFFIXES = ['폭투', '포일', '송구실책', '포구실책', '실책',
                    *sorted(NT_ERRORS)]

def _is_prefix_only(p):
    base = re.sub(r'\[.*?\]', '', p).strip()
    return any(pf in base for pf in _prefixes)

def _clean_result(value):
    if pd.isna(value):
        return value
    value = str(value).strip()
    value = value.replace('.', ',').replace(';', ',')
    if ',' not in value:
        return float('nan') if any(r in value for r in _remove_always) else value
    parts = [p.strip() for p in value.split(',')]
    if all(any(r in p for r in _remove_always) for p in parts):
        return parts[0]
    parts = [p for p in parts if not any(r in p for r in _remove_always)] or parts
    natout = next((p for p in parts if '낫아웃' in p), None)
    if natout:
        if natout == '낫아웃+':
            suf = next((p for p in parts if p != natout and any(s in p for s in _NATOUT_SUFFIXES)), None)
            if suf:
                return next((f'낫아웃+{s}' for s in _NATOUT_SUFFIXES if s in suf), '낫아웃+')
            return '낫아웃+'
        if any(s in natout for s in _NATOUT_SUFFIXES) or '낫아웃-' in natout:
            return natout
        return natout
    non_pref = [p for p in parts if not _is_prefix_only(p)]
    return non_pref[0] if non_pref else parts[0]

def step1_clean(df):
    solo_mask = df['result'].str.strip() == '대수비'
    n_del = solo_mask.sum()
    df = df[~solo_mask].copy().reset_index(drop=True)

    before = df['result'].copy()
    df['result'] = df['result'].apply(_clean_result)
    n_changed = (df['result'].astype(str) != before.astype(str)).sum()

    natout_bad = df[
        df['result'].astype(str).str.contains('낫아웃', na=False) &
        df['result'].astype(str).str.contains(',', na=False)
    ]
    print(f"  단독 대수비 삭제: {n_del}행 | 쉼표 처리: {n_changed}건", end='')

    issues = pd.DataFrame(columns=_ISSUE_COLS)
    if len(natout_bad):
        print(f"\n  ⚠ [낫아웃 미처리 의심] {len(natout_bad)}건 → {natout_bad['result'].tolist()}", end='')
        tmp = natout_bad[['game_id', 'result']].copy().rename(columns={'result': '상세'})
        tmp['단계'] = '[1] 쉼표 삭제'
        tmp['문제'] = '낫아웃 미처리 의심'
        issues = tmp[_ISSUE_COLS].reset_index(drop=True)
    print()
    return df, issues

# ════════════════════════════════════════════════════════════════
# Step 2: 재정의 (plate_seq)
#   이슈 반환: 타석 의심인데 plate_seq 없는 행 (game_id, result)
# ════════════════════════════════════════════════════════════════
_NON_BATTING_EXACT    = {'실책', '홈공과어필아웃'}
_NON_BATTING_CONTAINS = [
    '도루', '도루자', '주자아웃', '부정주루', '오버런', '런다운',
    '송구실책', '포구실책', *sorted(NT_ERRORS), '대주자', '견제사', '포일',
    '보크', '주루방해', '타구맞음', '루공과', '폭투', '득점', '리터치',
]

def _is_batting(result, batter=None):
    r = str(result).strip()
    if pd.isna(result) or r in ('nan', ''):
        return False
    r = r.split(',')[0].strip()
    if '낫아웃' in r:
        return True
    base = re.sub(r'\[.*?\]', '', r).strip()
    # 수비방해: 타자 본인의 단독 타석 결과(batter 있음)일 때는 실제 타격이므로
    # 정상적으로 plate_seq를 부여해야 하지만, 주자아웃처럼 별도 후속행
    # (batter 없음)으로 기록된 경우엔 타석이 아니므로 제외한다.
    if base == '수비방해':
        return pd.notna(batter)
    return base not in _NON_BATTING_EXACT and not any(k in base for k in _NON_BATTING_CONTAINS)

def step2_redefine(df):
    RENAME_MAP = {}
    for (team, old), new in RENAME_MAP.items():
        mask = (df['team'] == team) & (df['batter'] == old)
        if mask.any():
            df.loc[mask, 'batter'] = new

    df['plate_seq'] = None
    for (game_id, topbot), grp in df.groupby(['game_id', 'topbot'], sort=False):
        seq = 1
        for idx, row in grp.iterrows():
            if _is_batting(row['result'], row.get('batter')):
                df.at[idx, 'plate_seq'] = float(seq)
                seq += 1

    n_seq  = df['plate_seq'].notna().sum()
    _is_batting_row = df.apply(lambda r: _is_batting(r['result'], r.get('batter')), axis=1)
    no_seq = df[df['plate_seq'].isna() & df['result'].notna() & _is_batting_row]

    print(f"  plate_seq 부여: {n_seq}행", end='')

    issues = pd.DataFrame(columns=_ISSUE_COLS)
    if len(no_seq):
        gids = no_seq['game_id'].unique().tolist()
        vals = no_seq['result'].value_counts().head(3).to_dict()
        print(f"\n  ⚠ [plate_seq 누락 의심] {len(no_seq)}건 → game_id: {gids}\n    result 샘플: {vals}", end='')
        tmp = no_seq[['game_id', 'result']].copy().rename(columns={'result': '상세'})
        tmp['단계'] = '[2] 재정의'
        tmp['문제'] = 'plate_seq 누락 의심'
        issues = tmp[_ISSUE_COLS].reset_index(drop=True)
    print()
    return df, issues

# ════════════════════════════════════════════════════════════════
# Step 3: 행 채우기
#   이슈 반환: 채운 뒤에도 game_id/inning NaN 잔존 행
# ════════════════════════════════════════════════════════════════
_FILL_COLS = [
    'game_id', 'topbot', 'team', 'inning',
    'tournament', 'round', 'details', 'year', 'date', 'time', 'stadium',
    'team1', 'score1', 'result1', 'team2', 'score2', 'result2', 'def_team'
]

def step3_fill(df):
    fill_cols    = [c for c in _FILL_COLS if c in df.columns]
    daesubi_only = df['result'].str.strip() == '대수비'
    filled = skipped = 0

    for i in df.index[df['game_id'].isna()]:
        if daesubi_only.at[i]:
            skipped += 1
            continue
        prev = i - 1
        if prev < 0 or pd.isna(df.at[prev, 'game_id']):
            continue
        for col in fill_cols:
            df.at[i, col] = df.at[prev, col]
        filled += 1

    nan_game_id = int(df['game_id'].isna().sum())
    nan_inning  = int(df['inning'].isna().sum())

    print(f"  채운 행: {filled} | 스킵(대수비): {skipped}", end='')

    issues = pd.DataFrame(columns=_ISSUE_COLS)
    if nan_game_id or nan_inning:
        print(f"\n  ⚠ [game_id NaN 잔존] {nan_game_id}행 / [inning NaN 잔존] {nan_inning}행", end='')
        nan_rows = df[df['game_id'].isna()][['result']].copy()
        nan_rows['game_id'] = pd.NA
        nan_rows['단계'] = '[3] 행 채우기'
        nan_rows['문제'] = 'NaN 잔존 (game_id 미확인)'
        nan_rows['상세'] = nan_rows['result'].astype(str)
        issues = nan_rows[_ISSUE_COLS].reset_index(drop=True)
    print()
    return df, issues

# ════════════════════════════════════════════════════════════════
# Step 4: enter_inning_summary 계산
#   이슈 반환: enter_out=0인 이닝 (game_id, inning, topbot)
# ════════════════════════════════════════════════════════════════
def step4_summary(df):
    df['inning']       = pd.to_numeric(df['inning'],       errors='coerce').fillna(0).astype(int)
    df['outs_when_up'] = pd.to_numeric(df['outs_when_up'], errors='coerce').fillna(0).astype(int)

    def _get_score(row):
        col = 'away_score' if row['topbot'] == 'top' else 'home_score'
        v   = row.get(col, None)
        return v if pd.notna(v) else 0.0

    df['score_this_pa'] = pd.to_numeric(df.apply(_get_score, axis=1), errors='coerce').fillna(0.0)

    summary = df.groupby(['file_name', 'game_id', 'inning', 'topbot'], sort=False).agg(
        enter_out  =('outs_when_up',  'sum'),
        enter_score=('score_this_pa', 'sum')
    ).reset_index()
    summary['enter_out']   = summary['enter_out'].astype(int)
    summary['enter_score'] = summary['enter_score'].astype(int)

    n_games  = summary['game_id'].nunique()
    n_inn    = len(summary)
    zero_out = summary[summary['enter_out'] == 0]

    print(f"  {n_games}경기 / {n_inn}이닝 처리", end='')

    issues = pd.DataFrame(columns=_ISSUE_COLS)
    if len(zero_out):
        gids = zero_out['game_id'].unique().tolist()
        print(f"\n  ⚠ [enter_out=0 이닝] {len(zero_out)}건 → game_id: {gids}", end='')
        tmp = zero_out[['game_id', 'inning', 'topbot']].copy()
        tmp['단계'] = '[4] enter_summary'
        tmp['문제'] = 'enter_out=0'
        tmp['상세'] = tmp['inning'].astype(str) + '이닝 ' + tmp['topbot']
        issues = tmp[_ISSUE_COLS].reset_index(drop=True)
    print()
    return summary, issues

# ════════════════════════════════════════════════════════════════
# 스타일 엑셀 저장: enter_inning_summary
# ════════════════════════════════════════════════════════════════
def save_summary_xlsx(df, path):
    write_styled_xlsx(
        df, path, sheet="이닝별 요약",
        headers=['file_name', 'game_id', 'inning', 'topbot', 'enter_out', 'enter_score'],
        widths=[14, 12, 8, 8, 10, 12],
        emphasis=('file_name', 'game_id'))

# ════════════════════════════════════════════════════════════════
# 스타일 엑셀 저장: diff_inning_summary
# ════════════════════════════════════════════════════════════════
def save_diff_xlsx(df, path):
    write_styled_xlsx(
        df, path, sheet="이닝별 diff",
        headers=['file_name', 'game_id', 'inning', 'topbot',
                 'actual_score', 'enter_score', 'score_diff',
                 'real_out', 'enter_out', 'out_diff'],
        widths=[14, 12, 8, 8, 14, 12, 11, 10, 10, 10],
        emphasis=('file_name', 'game_id'),
        zero_green=('score_diff', 'out_diff'))

# ════════════════════════════════════════════════════════════════
# 스타일 엑셀 저장: issues (오타/이상 항목 목록)
#   헤더: 주황색 / 열: file_name, game_id, 단계, 문제, 상세
# ════════════════════════════════════════════════════════════════
def save_issues_xlsx(df, path):
    write_styled_xlsx(
        df, path, sheet="이슈 목록",
        headers=['file_name', 'game_id', '단계', '문제', '상세'],
        widths=[16, 10, 16, 22, 40],
        header_color=_C_ISSUES, left=('상세',), autofilter=True)

print("✓ 함수 정의 완료")

In [ ]:
# ════════════════════════════════════════════
# 메인 파이프라인 실행
# ════════════════════════════════════════════
# ── 파일 범위 지정 실행 헬퍼
#   특정 파일까지만  : end_file='2022_07_a'
#   특정 파일부터    : start_file='2022_04_a'
#   특정 구간만      : start_file='2022_04_a', end_file='2022_07_c'
def _slice_file_range(source_files, start_file=None, end_file=None):
    """정렬된 source_files를 file stem 기준 [start_file, end_file] 구간만 남긴다.
    start_file 생략 -> 처음부터 / end_file 생략 -> 끝까지 (양끝 포함).
    '2022_07_a' / '2022_07_a.xlsx' 둘 다 허용."""
    if not source_files or (not start_file and not end_file):
        return source_files
    stems = [os.path.splitext(os.path.basename(p))[0] for p in source_files]

    def _idx(name, label):
        s = os.path.splitext(str(name).strip())[0]
        if s not in stems:
            raise ValueError(f"{label}='{name}' 을(를) 찾을 수 없습니다. "
                             f"사용 가능: {stems[0]} ~ {stems[-1]} ({len(stems)}개)")
        return stems.index(s)

    i = _idx(start_file, 'start_file') if start_file else 0
    j = _idx(end_file, 'end_file') if end_file else len(stems) - 1
    if i > j:
        raise ValueError(f"범위 오류: start_file='{stems[i]}' 이 end_file='{stems[j]}' 보다 뒤입니다")
    return source_files[i:j + 1]

# ── 범위 실행 시 출력 파일명 접미사 (전체 결과 덮어쓰기 방지)
def _range_suffix(source_files, start_file=None, end_file=None):
    """범위 실행이면 '_04_b~07_a' 같은 접미사를, 전체 실행이면 '' 을 돌려준다."""
    if not (start_file or end_file) or not source_files:
        return ''

    def _short(p):
        st = os.path.splitext(os.path.basename(p))[0]
        pre = f"{YEAR}_"
        return st[len(pre):] if st.startswith(pre) else st

    s, e = _short(source_files[0]), _short(source_files[-1])
    return f"_{s}" if s == e else f"_{s}~{e}"

def _run_check_for_one_year(year, start_file=None, end_file=None):
    """한 연도치만 처리한다 — 여러 연도를 한 번에 돌리려면 run_check_pipeline()을 쓴다."""
    global YEAR, YEAR_DIR, REAL_PATH, SUMMARY_OUT, DIFF_OUT, ISSUES_OUT
    YEAR = year
    YEAR_DIR = f"{BASE_DIR}/{YEAR}"
    REAL_PATH   = f"{BASE_DIR}/inning_summary/inning_summary_{YEAR}.xlsx"
    SUMMARY_OUT = f"{YEAR_DIR}/검수/{YEAR}_enter_inning_summary.xlsx"
    DIFF_OUT    = f"{YEAR_DIR}/검수/{YEAR}_diff_inning_summary.xlsx"
    ISSUES_OUT  = f"{YEAR_DIR}/검수/{YEAR}_issues.xlsx"
    os.makedirs(f"{YEAR_DIR}/검수", exist_ok=True)

    # ── 소스 파일 수집: YEAR_DIR 바로 아래 N월 폴더만 탐색
    source_files = []
    month_pat = re.compile(r'^\d+월$')
    file_pat  = re.compile(rf'^{YEAR}_\d{{2}}_[a-z]+\.xlsx$')

    for entry in sorted(os.listdir(YEAR_DIR)):
        entry_nfc = unicodedata.normalize('NFC', entry)
        if not month_pat.match(entry_nfc):
            continue
        month_dir = os.path.join(YEAR_DIR, entry_nfc)
        if not os.path.isdir(month_dir):
            continue
        for f in sorted(os.listdir(month_dir)):
            f_nfc = unicodedata.normalize('NFC', f)
            if file_pat.match(f_nfc):
                source_files.append(os.path.join(month_dir, f_nfc))
    source_files.sort()
    _n_all = len(source_files)
    source_files = _slice_file_range(source_files, start_file, end_file)
    if len(source_files) != _n_all:
        _s = os.path.splitext(os.path.basename(source_files[0]))[0]
        _e = os.path.splitext(os.path.basename(source_files[-1]))[0]
        print(f"총 {_n_all}개 중 {len(source_files)}개만 처리: {_s} ~ {_e}\n")
    else:
        print(f"총 {len(source_files)}개 파일 발견\n")

    _sfx = _range_suffix(source_files, start_file, end_file)
    if _sfx:
        SUMMARY_OUT = f"{YEAR_DIR}/검수/{YEAR}_enter_inning_summary{_sfx}.xlsx"
        DIFF_OUT    = f"{YEAR_DIR}/검수/{YEAR}_diff_inning_summary{_sfx}.xlsx"
        ISSUES_OUT  = f"{YEAR_DIR}/검수/{YEAR}_issues{_sfx}.xlsx"
        print(f"  범위 실행 → 출력 파일명에 '{_sfx}' 접미사 적용 (전체 결과 보존)\n")

    all_summaries = []
    all_issues    = []   # 각 step에서 수집된 이슈 DataFrame 목록
    problem_files = []
    t_total = time.time()

    # ── 파일별 처리 루프
    for idx, fpath in enumerate(source_files, 1):
        fname    = unicodedata.normalize('NFC', os.path.basename(fpath))
        stem     = os.path.splitext(fname)[0]
        problems = []

        print("=" * 60)
        print(f"[{idx}/{len(source_files)}] {fname}")
        print("-" * 60)
        t0 = time.time()

        try:
            df = pd.read_excel(fpath)
            df['file_name'] = stem
            print(f"  로드: {len(df)}행")
        except Exception as e:
            print(f"  ❌ 로드 실패: {e}")
            problem_files.append((fname, [f"로드 실패: {e}"]))
            continue

        # Step 1: 쉼표 삭제
        print("[1] 쉼표 삭제")
        df, iss1 = step1_clean(df)
        if len(iss1):
            iss1.insert(0, 'file_name', stem)
            all_issues.append(iss1)
            problems.append(f"낫아웃 미처리 {len(iss1)}건")

        # Step 2: plate_seq 재정의
        print("[2] 재정의 (plate_seq)")
        df, iss2 = step2_redefine(df)
        if len(iss2):
            iss2.insert(0, 'file_name', stem)
            all_issues.append(iss2)
            problems.append(f"plate_seq 누락 {len(iss2)}건")

        # Step 3: 비타석 행 채우기
        print("[3] 행 채우기")
        df, iss3 = step3_fill(df)
        if len(iss3):
            iss3.insert(0, 'file_name', stem)
            all_issues.append(iss3)
            problems.append(f"NaN 잔존 {len(iss3)}행")

        # Step 4: 이닝별 요약 계산
        print("[4] enter_inning_summary")
        summary, iss4 = step4_summary(df)
        all_summaries.append(summary)
        if len(iss4):
            iss4.insert(0, 'file_name', stem)
            all_issues.append(iss4)
            problems.append(f"enter_out=0 이닝 {len(iss4)}건")

        elapsed = time.time() - t0
        status  = "✅" if not problems else "⚠"
        extra   = f" | 문제: {', '.join(problems)}" if problems else ""
        print(f"{status} {fname} 완료 ({elapsed:.1f}초){extra}")
        if problems:
            problem_files.append((fname, problems))

    # ── 전체 합본 → 스타일 엑셀 저장
    print("\n" + "=" * 60)
    print("enter_inning_summary 저장 중...")
    full = pd.concat(all_summaries, ignore_index=True)
    full['_tb'] = full['topbot'].map({'top': 0, 'bot': 1})
    full = full.sort_values(['game_id', '_tb', 'inning']).drop(columns='_tb').reset_index(drop=True)
    save_summary_xlsx(full, SUMMARY_OUT)
    print(f"  → {SUMMARY_OUT}")

    # ── real 파일과 대조
    print("real 파일 대조 중...")
    real = pd.read_excel(REAL_PATH)
    real['game_id'] = real['game_id'].ffill().astype(int)

    merged = pd.merge(
        real[['game_id', 'inning', 'topbot', 'score', 'real_out']].rename(columns={'score': 'actual_score'}),
        full[['file_name', 'game_id', 'inning', 'topbot', 'enter_out', 'enter_score']],
        on=['game_id', 'inning', 'topbot'],
        how='inner'
    )
    merged['score_diff'] = merged['enter_score'] - merged['actual_score']
    merged['out_diff']   = merged['enter_out']   - merged['real_out']
    merged = merged.reset_index(drop=True)
    save_diff_xlsx(merged, DIFF_OUT)
    print(f"  → {DIFF_OUT}")

    # ── 대조 결과도 issues에 추가
    # diff != 0 = 입력값과 실제값 불일치 → 해당 파일·경기 수정 필요
    score_bad = merged[merged['score_diff'] != 0]
    out_bad   = merged[merged['out_diff']   != 0]

    if len(score_bad):
        si = score_bad[['file_name', 'game_id', 'inning', 'topbot', 'score_diff']].copy()
        si['단계'] = '[대조] score'
        si['문제'] = 'score_diff != 0'
        si['상세'] = si['inning'].astype(str) + '이닝 ' + si['topbot'] + ' (diff=' + si['score_diff'].astype(str) + ')'
        all_issues.append(si[['file_name', 'game_id', '단계', '문제', '상세']])

    if len(out_bad):
        oi = out_bad[['file_name', 'game_id', 'inning', 'topbot', 'out_diff']].copy()
        oi['단계'] = '[대조] out'
        oi['문제'] = 'out_diff != 0'
        oi['상세'] = oi['inning'].astype(str) + '이닝 ' + oi['topbot'] + ' (diff=' + oi['out_diff'].astype(str) + ')'
        all_issues.append(oi[['file_name', 'game_id', '단계', '문제', '상세']])

    # ── issues 파일 저장
    # file_name 열로 원본 파일(예: 2026/4월/2026_04_a.xlsx) 바로 찾아갈 수 있음
    print("issues 파일 저장 중...")
    if all_issues:
        issues_df = pd.concat(all_issues, ignore_index=True)
        issues_df = issues_df[['file_name', 'game_id', '단계', '문제', '상세']]
        save_issues_xlsx(issues_df, ISSUES_OUT)
        print(f"  → {ISSUES_OUT} ({len(issues_df)}건)")
    else:
        print("  ✅ 이슈 없음 — issues 파일 미생성")

    # ── 최종 요약 출력
    total_t = time.time() - t_total

    print("\n" + "=" * 60)
    print("전체 요약")
    print("-" * 60)
    print(f"처리 파일: {len(source_files)}개 | 소요: {total_t:.1f}초")

    if problem_files:
        print(f"\n⚠ 문제 발생 파일: {len(problem_files)}개")
        for fname, probs in problem_files:
            print(f"  - {fname}: {', '.join(probs)}")
    else:
        print("\n✅ 모든 파일 정상 처리")

    print(f"\n[최종 대조] 매칭 경기: {merged['game_id'].nunique():,}개")
    print(f"  score_diff != 0: {len(score_bad)}건 → {sorted(score_bad['game_id'].unique().tolist())}")
    print(f"  out_diff   != 0: {len(out_bad)}건 → {sorted(out_bad['game_id'].unique().tolist())}")
    print("=" * 60)

    return len(score_bad), len(out_bad)


def run_check_pipeline(years=2022, start_file=None, end_file=None):
    """years에 정수 하나(예: 2022) 또는 리스트(예: [2020, 2021, 2022])를 넣으면
    그 연도(들)를 순서대로 다 처리하고, 맨 끝에 연도별 score/out 불일치 건수
    요약표를 보여준다. 예전처럼 인자 없이 run_check_pipeline()만 불러도
    기본값(2022) 한 해만 처리한다."""
    years_list = list(years) if isinstance(years, (list, tuple, set)) else [years]
    if (start_file or end_file) and len(years_list) > 1:
        raise ValueError(f"start_file/end_file 범위 지정은 단일 연도에서만 쓸 수 있습니다 (years={years_list}).")

    summary_rows = []
    for y in years_list:
        print("\n" + "#" * 70)
        print(f"### {y}년 처리 시작")
        print("#" * 70)
        s_n, o_n = _run_check_for_one_year(y, start_file, end_file)
        summary_rows.append({'연도': y, 'score 불일치': s_n, 'out 불일치': o_n})

    print("\n" + "=" * 60)
    print("연도별 요약")
    print("=" * 60)
    summary_df = pd.DataFrame(summary_rows)
    display(summary_df)
    return summary_df


# ────────────────────────────────────────────────────────────────────
# run_check_pipeline() 실행 예시
# ────────────────────────────────────────────────────────────────────
#   run_check_pipeline(2022)                       # 2022년 전체 (기존과 동일)
#   run_check_pipeline([2020, 2021, 2022])         # 여러 연도 전체
#
#   ── 특정 파일까지만 / 특정 구간만 ──
#   run_check_pipeline(2022, end_file='2022_07_a')                          # 처음 ~ 07_a 까지
#   run_check_pipeline(2022, start_file='2022_04_b')                        # 04_b ~ 끝까지
#   run_check_pipeline(2022, start_file='2022_04_b', end_file='2022_07_a')  # 04_b ~ 07_a 구간만
#   run_check_pipeline(2022, start_file='2022_07_a', end_file='2022_07_a')  # 그 파일 하나만
#
#   ── 위치 인자로 짧게 써도 된다 (years, start_file, end_file 순서) ──
#   run_check_pipeline(2022, '2022_04_b', '2022_07_a')   # 04_b ~ 07_a 구간만
#   run_check_pipeline(2022, '2022_04_b')                # 04_b ~ 끝까지
#     단, end_file만 줄 땐 키워드로: run_check_pipeline(2022, end_file='2022_07_a')
#
#   · 파일명은 '2022_07_a' / '2022_07_a.xlsx' 둘 다 인식
#   · 범위 지정 시 years는 단일 연도만 (여러 연도 + 범위 → ValueError)
#   · 없는 파일명이나 start>end 이면 ValueError로 알려준다
#   · 범위 실행하면 출력 파일명에 '_04_b~07_a' 접미사가 붙어
#     전체 실행 결과 파일을 덮어쓰지 않는다
# ────────────────────────────────────────────────────────────────────
# run_check_pipeline([2020, 2021, 2022, 2026])
# run_check_pipeline([2020, 2021])
# run_check_pipeline(2023, '2023_07_a')
# run_check_pipeline(2024, '2024_03_a', '2024_04_n')
# run_check_pipeline(2024, '2024_07_a')
# run_check_pipeline(2025, '2025_03_a', '2025_03_v')
# run_check_pipeline(2025, '2025_07_a')
# 반환값을 변수로 받는다 - 그냥 호출하면 Jupyter 가 반환된 표를 한 번 더 그려서
# 같은 표가 두 번 나온다.
summary = run_check_pipeline(2026)